# Calimerge — `keypoints_3d.raw.npz` viewer

Standalone teaching notebook. Pick a `keypoints_3d.raw.npz` file dumped by Calimerge during a recording and inspect:

1. The full skeleton at a few representative frames
2. Foot position over time
3. Inter-frame dt (frame-skip diagnostic)

**No imports from `calimerge` itself** — only `numpy`, `matplotlib`, and the standard library. The SynthPose-52 keypoint schema is inlined below, so this notebook keeps working even if the calimerge package is uninstalled / refactored / renamed in the future.

## 1. Pick the .npz via a file browser

In [ ]:
from pathlib import Path
import tkinter as tk
from tkinter import filedialog

def pick_npz(default_dir: str | None = None) -> Path:
    """Pop a native file dialog and return the chosen .npz path.

    Falls back to a hard-coded path if the user cancels.
    """
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    chosen = filedialog.askopenfilename(
        title="Pick keypoints_3d.raw.npz",
        initialdir=default_dir or str(Path.home()),
        filetypes=[("NumPy archive", "*.npz"), ("All files", "*.*")],
    )
    root.destroy()
    if not chosen:
        raise FileNotFoundError("No file selected.")
    return Path(chosen)

# Default location: ~/Documents/Calimerge or whichever workout dir the
# user has Calimerge configured to write into.
default_dir = str(Path.home() / "Documents" / "Calimerge")
if not Path(default_dir).is_dir():
    default_dir = None

npz_path = pick_npz(default_dir=default_dir)
print(f"Loaded: {npz_path}")

## 2. SynthPose-52 keypoint schema (inlined)

Calimerge writes 52 keypoints per person per frame in this fixed order. Indices 0–16 are the COCO-17 set; 17–51 are anatomical landmarks added by the SynthPose model. Side classification (`L` / `R` / `C`) is what makes left-vs-right plots possible.

In [ ]:
SYNTHPOSE_MARKERS = {
    0: "Nose", 1: "L_Eye", 2: "R_Eye", 3: "L_Ear", 4: "R_Ear",
    5: "L_Shoulder", 6: "R_Shoulder", 7: "L_Elbow", 8: "R_Elbow",
    9: "L_Wrist", 10: "R_Wrist", 11: "L_Hip", 12: "R_Hip",
    13: "L_Knee", 14: "R_Knee", 15: "L_Ankle", 16: "R_Ankle",
    17: "sternum", 18: "rshoulder", 19: "lshoulder", 20: "r_lelbow",
    21: "l_lelbow", 22: "r_melbow", 23: "l_melbow", 24: "r_lwrist",
    25: "l_lwrist", 26: "r_mwrist", 27: "l_mwrist", 28: "r_ASIS",
    29: "l_ASIS", 30: "r_PSIS", 31: "l_PSIS", 32: "r_knee",
    33: "l_knee", 34: "r_mknee", 35: "l_mknee", 36: "r_ankle",
    37: "l_ankle", 38: "r_mankle", 39: "l_mankle", 40: "r_5meta",
    41: "l_5meta", 42: "r_toe", 43: "l_toe", 44: "r_big_toe",
    45: "l_big_toe", 46: "l_calc", 47: "r_calc", 48: "C7",
    49: "L2", 50: "T11", 51: "T6",
}

def side_of(idx: int) -> str:
    """Anatomical side: 'L' / 'R' / 'C' (center)."""
    name = SYNTHPOSE_MARKERS.get(idx, "")
    if name.startswith(("L_", "l_")):
        return "L"
    if name.startswith(("R_", "r_")):
        return "R"
    return "C"

# Index aliases — handy for slicing later.
L_HIP, R_HIP = 11, 12
L_ANKLE, R_ANKLE = 15, 16

# Bones to draw between keypoint pairs. COCO-17 connectivity plus a few
# spine / foot strokes from the SynthPose extension.
SKELETON_BONES = [
    # Head
    (0, 1), (0, 2), (1, 3), (2, 4),
    # Shoulders / arms
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    # Torso
    (5, 11), (6, 12), (11, 12),
    # Legs
    (11, 13), (13, 15), (12, 14), (14, 16),
    # Feet
    (15, 41), (15, 43), (15, 46),  # left foot landmarks
    (16, 40), (16, 42), (16, 47),  # right foot landmarks
    # Spine
    (0, 48), (48, 51), (51, 50), (50, 49),
]

print(f"{len(SYNTHPOSE_MARKERS)} keypoints, {len(SKELETON_BONES)} bones drawn")

## 3. Load the file

Calimerge's `.npz` schema:
- `timestamps` — `(n_frames,)` float64 seconds since recording start
- `keypoints_3d` — `(n_frames, n_persons, 52, 3)` float32; NaN for missing
- `person_count` — `(n_frames,)` int32 number of valid persons that frame
- `primary_person_index` — `(n_frames,)` int32 index of "the" person

In [ ]:
import numpy as np

data = np.load(npz_path)
times = data["timestamps"]                  # (T,)
kps = data["keypoints_3d"]                  # (T, P, K, 3)
counts = data["person_count"]               # (T,)
primary = data.get("primary_person_index", np.zeros(len(times), dtype=np.int32))

n_frames, max_persons, n_kps, _ = kps.shape
duration = float(times[-1] - times[0]) if n_frames > 1 else 0.0
fps = n_frames / duration if duration > 0 else float("nan")

print(f"frames={n_frames} | persons<= {max_persons} | keypoints={n_kps} | duration={duration:.2f} s | mean fps={fps:.1f}")
print(f"per-frame valid-person counts: min={counts.min()} max={counts.max()} mean={counts.mean():.2f}")

if n_kps != 52:
    print(f"WARNING: file has {n_kps} keypoints, this notebook assumes SynthPose-52.")

## 4. Whole skeleton at a few frames

Three views (start / middle / end) of the primary person. Left-side keypoints draw blue, right-side red, midline gray — same convention as the live calimerge viewer.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the projection)

SIDE_COLOR = {"L": "#5099ff", "R": "#ff5050", "C": "#aaaaaa"}

def plot_skeleton(ax, frame_kps: np.ndarray, title: str = "") -> None:
    """Draw one person's skeleton from a (52, 3) array. NaNs are skipped."""
    valid = ~np.isnan(frame_kps).any(axis=1)

    # Bones — colour by majority side; if mixed, use center gray.
    for i, j in SKELETON_BONES:
        if not (valid[i] and valid[j]):
            continue
        si, sj = side_of(i), side_of(j)
        if si == sj:
            color = SIDE_COLOR[si]
        elif "C" in (si, sj):
            color = SIDE_COLOR[si if si != "C" else sj]
        else:
            color = SIDE_COLOR["C"]
        xs = [frame_kps[i, 0], frame_kps[j, 0]]
        ys = [frame_kps[i, 1], frame_kps[j, 1]]
        zs = [frame_kps[i, 2], frame_kps[j, 2]]
        ax.plot(xs, ys, zs, color=color, linewidth=2)

    # Joints
    for k in range(len(frame_kps)):
        if not valid[k]:
            continue
        ax.scatter(
            frame_kps[k, 0], frame_kps[k, 1], frame_kps[k, 2],
            color=SIDE_COLOR[side_of(k)], s=18, depthshade=True,
        )

    ax.set_title(title)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_zlabel("z (m)")

# Pick frames where the primary person actually has data.
valid_frames = [i for i in range(n_frames) if counts[i] > 0]
if not valid_frames:
    raise RuntimeError("No frames contain any valid person — nothing to plot.")

picks = [
    valid_frames[0],
    valid_frames[len(valid_frames) // 2],
    valid_frames[-1],
]

fig = plt.figure(figsize=(15, 5))
for col, fi in enumerate(picks):
    pi = int(primary[fi]) if primary[fi] < counts[fi] else 0
    ax = fig.add_subplot(1, 3, col + 1, projection="3d")
    plot_skeleton(ax, kps[fi, pi], title=f"frame {fi}  t={times[fi]:.2f} s")

fig.suptitle("Primary person — start / middle / end")
fig.tight_layout()
plt.show()

## 5. Foot position over time

Plots ankle x / y / z (in metres) for left + right foot of the primary person. The vertical (z) trace makes step cycles obvious — peaks during swing, plateaus during stance.

In [ ]:
# Pull the primary-person ankle traces out across all frames.
left_ankle = np.full((n_frames, 3), np.nan, dtype=np.float32)
right_ankle = np.full((n_frames, 3), np.nan, dtype=np.float32)
for fi in range(n_frames):
    if counts[fi] == 0:
        continue
    pi = int(primary[fi]) if primary[fi] < counts[fi] else 0
    left_ankle[fi] = kps[fi, pi, L_ANKLE]
    right_ankle[fi] = kps[fi, pi, R_ANKLE]

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, axis_idx, axis_label in zip(axes, range(3), ("x (m)", "y (m)", "z (m)")):
    ax.plot(times, left_ankle[:, axis_idx], color=SIDE_COLOR["L"], label="L_Ankle")
    ax.plot(times, right_ankle[:, axis_idx], color=SIDE_COLOR["R"], label="R_Ankle")
    ax.set_ylabel(axis_label)
    ax.grid(True, alpha=0.3)
axes[0].set_title("Ankle position over time (primary person)")
axes[-1].set_xlabel("time (s)")
axes[0].legend(loc="upper right")
fig.tight_layout()
plt.show()

## 6. Inter-frame dt — frame-skip diagnostic

If the camera dropped frames mid-recording, dt spikes show up here. Expected steady-state value is `1 / fps` for the recording (e.g. ~0.033 s at 30 fps). Spikes ≥ 2× the median are usually the cameras choking under load or the detection worker stalling.

In [ ]:
dts = np.diff(times)
median_dt = float(np.median(dts)) if len(dts) else float("nan")
skip_threshold = 2.0 * median_dt
skips = np.where(dts > skip_threshold)[0]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(times[1:], dts * 1000.0, color="#444", linewidth=1, label="dt (ms)")
ax.axhline(median_dt * 1000.0, color="#4caf50", linestyle="--",
           label=f"median {median_dt * 1000:.1f} ms")
ax.axhline(skip_threshold * 1000.0, color="#ff5050", linestyle=":",
           label=f"skip threshold {skip_threshold * 1000:.1f} ms")
for s in skips:
    ax.axvspan(times[s], times[s + 1], color="#ff5050", alpha=0.15)
ax.set_xlabel("time (s)")
ax.set_ylabel("frame interval (ms)")
ax.set_title(f"Inter-frame dt — {len(skips)} apparent skip(s) of >{skip_threshold * 1000:.0f} ms")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f"frames: {n_frames}, total duration: {duration:.2f} s")
print(f"median dt: {median_dt * 1000:.2f} ms  -> ~{1.0 / median_dt:.1f} fps" if median_dt else "")
print(f"max dt:    {dts.max() * 1000:.2f} ms" if len(dts) else "")
print(f"#skips (dt > 2x median): {len(skips)}")